# ECG-FM Stage 2 — Kaggle One-Click

### Before running:

**1. Add PTB-XL dataset** (right sidebar → Add Data → search `ptb-xl-dataset` by khyeh0719)

**2. Update your `ecgfm-models` Kaggle dataset** to include these 4 files:
- `mimic_iv_ecg_physionet_pretrained.pt` — from local `models/ecgfm/`
- `ecgfm_stage1.pt` — from local `models/`
- `ecgfm_finetune.py` — from local repo root
- `ecgfm_encoder.py` — from local repo root

Then add it to this notebook: **Add Data → Your Datasets → ecgfm-models**

**3. Enable GPU:** right sidebar → Accelerator → GPU T4 x2

**4. Run the single cell below** — that's it.

---
Output model `ecgfm_stage2.pt` will appear in the **Output tab** automatically.

In [ ]:
# ── Single cell: setup + train + show results ────────────────────────────────
import os, shutil, torch, psutil
from pathlib import Path

# ── 1. Check GPU ─────────────────────────────────────────────────────────────
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    raise RuntimeError('No GPU detected — enable T4 GPU in the right sidebar before running.')
print('RAM:', round(psutil.virtual_memory().total / 1e9, 1), 'GB\n')

# ── 2. Auto-detect input datasets ────────────────────────────────────────────
PTBXL_PATH = None
MODEL_PATH  = None

for d in os.listdir('/kaggle/input'):
    candidate = f'/kaggle/input/{d}'
    if os.path.exists(f'{candidate}/ptbxl_database.csv'):
        PTBXL_PATH = candidate
    if os.path.exists(f'{candidate}/ecgfm_stage1.pt'):
        MODEL_PATH = candidate

if not PTBXL_PATH:
    raise FileNotFoundError('PTB-XL not found — add ptb-xl-dataset via Add Data.')
if not MODEL_PATH:
    raise FileNotFoundError('ecgfm-models not found — add your dataset via Add Data.')

for fname in ['ecgfm_finetune.py', 'ecgfm_encoder.py',
              'mimic_iv_ecg_physionet_pretrained.pt', 'ecgfm_stage1.pt']:
    if not os.path.exists(f'{MODEL_PATH}/{fname}'):
        raise FileNotFoundError(
            f'Missing {fname} in ecgfm-models dataset.\n'
            f'Upload it to your Kaggle dataset and re-add the dataset to this notebook.'
        )

n_dat = len(list(Path(PTBXL_PATH).rglob('*.dat')))
print(f'PTB-XL:      {PTBXL_PATH}  ({n_dat} signal files)')
print(f'Model files: {MODEL_PATH}\n')
if n_dat < 20000:
    print(f'WARNING: only {n_dat}/21837 signal files — check the dataset includes records500/')

# ── 3. Set up working directory ───────────────────────────────────────────────
WORKDIR = '/kaggle/working/ecgfm'
os.makedirs(f'{WORKDIR}/models/ecgfm', exist_ok=True)
os.makedirs(f'{WORKDIR}/ekg_datasets', exist_ok=True)

# Copy scripts
shutil.copy(f'{MODEL_PATH}/ecgfm_finetune.py', f'{WORKDIR}/ecgfm_finetune.py')
shutil.copy(f'{MODEL_PATH}/ecgfm_encoder.py',  f'{WORKDIR}/ecgfm_encoder.py')

# Copy model checkpoints
shutil.copy(f'{MODEL_PATH}/mimic_iv_ecg_physionet_pretrained.pt',
            f'{WORKDIR}/models/ecgfm/mimic_iv_ecg_physionet_pretrained.pt')
shutil.copy(f'{MODEL_PATH}/ecgfm_stage1.pt',
            f'{WORKDIR}/models/ecgfm_stage1.pt')

# Symlink PTB-XL
link = f'{WORKDIR}/ekg_datasets/ptbxl'
if os.path.islink(link):
    os.remove(link)
os.symlink(os.path.abspath(PTBXL_PATH), link)

print('Setup complete:')
print(f'  ecgfm_finetune.py  — {round(os.path.getsize(WORKDIR+"/ecgfm_finetune.py")/1e3)} KB')
print(f'  encoder pretrained — {round(os.path.getsize(WORKDIR+"/models/ecgfm/mimic_iv_ecg_physionet_pretrained.pt")/1e6)} MB')
print(f'  stage1 checkpoint  — {round(os.path.getsize(WORKDIR+"/models/ecgfm_stage1.pt")/1e6)} MB')
print(f'  PTB-XL symlink     → {os.path.realpath(link)}\n')

# ── 4. Install dependencies ───────────────────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'wfdb', 'scipy', 'scikit-learn'],
               check=True)
print('Dependencies installed\n')

# ── 5. Run Stage 2 training ───────────────────────────────────────────────────
# Expected: ~20 min/epoch, ~3 hours total (early stopping patience=8)
# Checkpoint saved to models/ecgfm_stage2.pt at every validation improvement
#
# If CUDA out-of-memory: change --batch_s2 8  to  --batch_s2 4
os.chdir(WORKDIR)
!python -u ecgfm_finetune.py --stage 2 --batch_s2 8

# ── 6. Show results ───────────────────────────────────────────────────────────
model_path = f'{WORKDIR}/models/ecgfm_stage2.pt'
if os.path.exists(model_path):
    print(f'\necgfm_stage2.pt saved ({round(os.path.getsize(model_path)/1e6)} MB)')
    print('Download: Output tab (right sidebar) → ecgfm/models/ecgfm_stage2.pt\n')
    ckpt = torch.load(model_path, map_location='cpu', weights_only=False)
    tm   = ckpt.get('test_metrics', {})
    print('Final test results:')
    print(f"  Accuracy : {tm.get('acc',  0):.1%}")
    print(f"  HYP F1   : {tm.get('hyp_f1',   0):.3f}  (baseline 0.375)")
    print(f"  Macro F1 : {tm.get('macro_f1', 0):.3f}  (baseline 0.492)")
else:
    print('WARNING: ecgfm_stage2.pt not found — training may not have completed')